# Random Forest Modeling + SHAP Analysis — Claim Prediction| Split | Months | Purpose ||---|---|---|| **Train** | All before 2024-11 | Fit model (with SMOTE) || **Validate** | 2024-12, 2025-01 | Evaluate model performance || **Predict** | 2025-02 | Generate predictions + SHAP values |

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, precision_recall_curve,
    average_precision_score, confusion_matrix, f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap
print("Imports OK.")

## 1. Configuration

In [ ]:
DATA_DIR = r'C:\Users\whita\Documents\case-tricura\output_data'
OUTPUT_DIR = r'C:\Users\whita\Documents\case-tricura\output_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLAIM_TYPES = {
    'fall': 'target_fall', 'wound': 'target_wound',
    'medication_error': 'target_medication_error', 'elopement': 'target_elopement',
    'altercation': 'target_altercation', 'choking': 'target_choking',
    'rth': 'target_rth',
}

EXCLUDE_COLS = [
    'resident_id', 'facility_id', 'year_month',
    'target_fall', 'target_wound', 'target_medication_error',
    'target_elopement', 'target_altercation', 'target_choking', 'target_rth',
]

RF_PARAMS = {
    'n_estimators': 500, 'max_depth': 15, 'min_samples_split': 20,
    'min_samples_leaf': 10, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1,
}

RANDOM_STATE = 42

# TIME SPLITS
TRAIN_CUTOFF = pd.Timestamp('2024-11-01')
VAL_MONTHS = [pd.Timestamp('2024-12-01'), pd.Timestamp('2025-01-01')]
PREDICT_MONTH = pd.Timestamp('2025-02-01')

print(f"Train:    months before {TRAIN_CUTOFF.strftime('%Y-%m')}")
print(f"Validate: {', '.join(m.strftime('%Y-%m') for m in VAL_MONTHS)}")
print(f"Predict:  {PREDICT_MONTH.strftime('%Y-%m')}")

## 2. Helper Functions

In [ ]:
def prepare_features(df, target_col):
    drop_cols = [c for c in EXCLUDE_COLS if c in df.columns]
    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    X = X.fillna(0).select_dtypes(include=[np.number]).replace([np.inf, -np.inf], 0)
    return X, y, X.columns.tolist()

def compute_normalized_shap(shap_values, feature_names):
    shap_raw = pd.DataFrame(shap_values, columns=feature_names)
    shap_abs = np.abs(shap_values)
    row_sums = shap_abs.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    shap_pct = (shap_abs / row_sums) * 100
    return pd.DataFrame(shap_pct, columns=feature_names), shap_raw

def print_model_report(y_true, y_pred, y_proba, claim_name):
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0, labels=[0, 1],
                                target_names=['No Claim', 'Claim']))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"  Confusion Matrix:  TN={cm[0,0]:,}  FP={cm[0,1]:,}  |  FN={cm[1,0]:,}  TP={cm[1,1]:,}")
    if len(np.unique(y_true)) > 1:
        print(f"  ROC AUC: {roc_auc_score(y_true, y_proba):.4f}")
        print(f"  Avg Precision: {average_precision_score(y_true, y_proba):.4f}")
    else:
        print(f"  (Only one class in set - AUC not computed)")

print("Helpers defined.")

## 3. Train, Validate & Predict — All Claim TypesFor each claim:1. **Train** on months < 2024-11 (with SMOTE balancing)2. **Validate** on 2024-12 + 2025-01 (report metrics)3. **Predict** on 2025-02 (save predictions + SHAP)

In [ ]:
all_model_summaries = []

for claim_key, target_col in CLAIM_TYPES.items():
    print(f"\n{'='*70}")
    print(f"  CLAIM TYPE: {claim_key.upper()}")
    print(f"{'='*70}")
    
    # Load
    filepath = os.path.join(DATA_DIR, f'claims_{claim_key}_monthly.parquet')
    df = pd.read_parquet(filepath)
    print(f"\n  Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
    
    # Split
    df_train = df[df['year_month'] < TRAIN_CUTOFF]
    df_val = df[df['year_month'].isin(VAL_MONTHS)]
    df_predict = df[df['year_month'] == PREDICT_MONTH]
    
    print(f"  Train:    {len(df_train):,} rows | Target: {df_train[target_col].value_counts().to_dict()}")
    print(f"  Validate: {len(df_val):,} rows | Target: {df_val[target_col].value_counts().to_dict()}")
    print(f"  Predict:  {len(df_predict):,} rows | Target: {df_predict[target_col].value_counts().to_dict()}")
    
    # Features
    X_train, y_train, feature_names = prepare_features(df_train, target_col)
    X_val, y_val, _ = prepare_features(df_val, target_col)
    X_predict, y_predict, _ = prepare_features(df_predict, target_col)
    X_val = X_val.reindex(columns=feature_names, fill_value=0)
    X_predict = X_predict.reindex(columns=feature_names, fill_value=0)
    print(f"  Features: {len(feature_names)}")
    
    # CV with SMOTE
    pos_count = int(y_train.sum())
    N_FOLDS = 5
    min_pos_in_fold = int(np.floor(pos_count * (N_FOLDS - 1) / N_FOLDS))
    smote_k_cv = min(5, min_pos_in_fold - 1) if min_pos_in_fold > 1 else 1
    smote_k_full = min(5, pos_count - 1) if pos_count > 1 else 1
    use_smote = pos_count >= 3
    
    if pos_count >= 10 and smote_k_cv >= 1 and use_smote:
        print(f"\n  {N_FOLDS}-fold CV with SMOTE (k_cv={smote_k_cv})...")
        pipeline_cv = ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k_cv)),
            ('rf', RandomForestClassifier(**RF_PARAMS)),
        ])
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        cv_scores = cross_val_score(pipeline_cv, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        cv_f1 = cross_val_score(pipeline_cv, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
        print(f"\n  CV ROC AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
        print(f"  CV F1:      {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
    elif pos_count >= 10:
        print(f"\n  {N_FOLDS}-fold CV (class_weight=balanced)...")
        model_cv = RandomForestClassifier(**RF_PARAMS, class_weight='balanced')
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        cv_scores = cross_val_score(model_cv, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        print(f"\n  CV ROC AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    else:
        cv_scores = None
        print(f"\n  (Skipping CV - only {pos_count} positive samples)")
    
    if use_smote:
        print(f"\n  SMOTE (k={smote_k_full}) + RF training...")
        smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k_full)
        X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
        print(f"  {y_train.value_counts().to_dict()} -> {pd.Series(y_train_res).value_counts().to_dict()}")
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_train_res, y_train_res)
    else:
        print(f"\n  Too few positives ({pos_count}) for SMOTE - class_weight='balanced'")
        model = RandomForestClassifier(**RF_PARAMS, class_weight='balanced')
        model.fit(X_train, y_train)
    
    # Validate
    print(f"\n  --- VALIDATION (2024-12 + 2025-01) ---")
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    print_model_report(y_val, y_val_pred, y_val_proba, claim_key)
    
    val_predictions = df_val[['resident_id', 'facility_id', 'year_month']].copy().reset_index(drop=True)
    val_predictions['claim_type'] = claim_key
    val_predictions['y_true'] = y_val.values
    val_predictions['y_pred'] = y_val_pred
    val_predictions['y_proba'] = y_val_proba
    val_predictions.to_parquet(os.path.join(OUTPUT_DIR, f'validation_{claim_key}.parquet'), index=False)
    
    # Predict 2025-02
    print(f"\n  --- PREDICTION (2025-02) ---")
    y_pred = model.predict(X_predict)
    y_proba = model.predict_proba(X_predict)[:, 1]
    if len(df_predict) > 0:
        print(f"  Predictions: {len(y_pred):,} | Predicted positive: {y_pred.sum():,} ({y_pred.mean()*100:.2f}%)")
        if len(np.unique(y_predict)) > 1:
            print_model_report(y_predict, y_pred, y_proba, claim_key)
    
    # Feature importance
    feat_imp = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    print(f"\n  Top 15 features (RF):")
    for _, row in feat_imp.head(15).iterrows():
        print(f"    {row['importance']:.4f}  {row['feature']}")
    
    # SHAP on prediction month
    print(f"\n  Computing SHAP for {len(X_predict):,} predictions...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_predict)
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_vals = shap_values[:, :, 1]
    else:
        shap_vals = shap_values
    
    shap_norm, shap_raw = compute_normalized_shap(shap_vals, feature_names)
    
    global_shap = pd.DataFrame({
        'feature': feature_names,
        'mean_abs_shap': np.abs(shap_vals).mean(axis=0),
        'mean_norm_shap_pct': shap_norm.mean(axis=0).values,
    }).sort_values('mean_abs_shap', ascending=False)
    
    print(f"\n  Top 15 features (norm SHAP %):")
    for _, row in global_shap.head(15).iterrows():
        print(f"    {row['mean_norm_shap_pct']:6.2f}%  {row['feature']}")
    
    # Save all outputs
    predictions = df_predict[['resident_id', 'facility_id', 'year_month']].copy().reset_index(drop=True)
    predictions['claim_type'] = claim_key
    predictions['y_true'] = y_predict.values
    predictions['y_pred'] = y_pred
    predictions['y_proba'] = y_proba
    predictions.to_parquet(os.path.join(OUTPUT_DIR, f'predictions_{claim_key}.parquet'), index=False)
    
    shap_norm_out = df_predict[['resident_id', 'facility_id', 'year_month']].copy().reset_index(drop=True)
    shap_norm_out['claim_type'] = claim_key
    shap_norm_out = pd.concat([shap_norm_out, shap_norm.reset_index(drop=True)], axis=1)
    shap_norm_out.to_parquet(os.path.join(OUTPUT_DIR, f'shap_normalized_{claim_key}.parquet'), index=False)
    
    shap_raw_out = df_predict[['resident_id', 'facility_id', 'year_month']].copy().reset_index(drop=True)
    shap_raw_out['claim_type'] = claim_key
    shap_raw_out = pd.concat([shap_raw_out, shap_raw.reset_index(drop=True)], axis=1)
    shap_raw_out.to_parquet(os.path.join(OUTPUT_DIR, f'shap_raw_{claim_key}.parquet'), index=False)
    
    global_shap['claim_type'] = claim_key
    global_shap.to_parquet(os.path.join(OUTPUT_DIR, f'shap_global_{claim_key}.parquet'), index=False)
    feat_imp['claim_type'] = claim_key
    feat_imp.to_parquet(os.path.join(OUTPUT_DIR, f'feature_importance_{claim_key}.parquet'), index=False)
    
    model_path = os.path.join(OUTPUT_DIR, f'model_rf_{claim_key}.joblib')
    joblib.dump({
        'model': model, 'feature_names': feature_names, 'claim_type': claim_key,
        'target_col': target_col, 'rf_params': RF_PARAMS,
        'train_cutoff': TRAIN_CUTOFF.strftime('%Y-%m'),
        'validation_months': [m.strftime('%Y-%m') for m in VAL_MONTHS],
        'prediction_month': PREDICT_MONTH.strftime('%Y-%m'),
    }, model_path)
    print(f"\n  All files saved for {claim_key}.")
    
    # Summary
    summary = {
        'claim_type': claim_key, 'train_rows': len(df_train), 'val_rows': len(df_val),
        'predict_rows': len(df_predict), 'train_positive': int(y_train.sum()),
        'val_positive': int(y_val.sum()), 'n_features': len(feature_names),
        'val_f1': f1_score(y_val, y_val_pred, zero_division=0),
        'prediction_month': PREDICT_MONTH.strftime('%Y-%m'),
    }
    if len(np.unique(y_val)) > 1:
        summary['val_auc'] = roc_auc_score(y_val, y_val_proba)
    else:
        summary['val_auc'] = None
    summary['cv_auc_mean'] = cv_scores.mean() if cv_scores is not None else None
    all_model_summaries.append(summary)

print(f"\n\n{'='*70}")
print("  ALL MODELS TRAINED!")
print(f"{'='*70}")

## 4. Combine & Save

In [ ]:
all_pred, all_val, all_gs, all_fi = [], [], [], []
for ck in CLAIM_TYPES:
    all_pred.append(pd.read_parquet(os.path.join(OUTPUT_DIR, f'predictions_{ck}.parquet')))
    all_val.append(pd.read_parquet(os.path.join(OUTPUT_DIR, f'validation_{ck}.parquet')))
    all_gs.append(pd.read_parquet(os.path.join(OUTPUT_DIR, f'shap_global_{ck}.parquet')))
    all_fi.append(pd.read_parquet(os.path.join(OUTPUT_DIR, f'feature_importance_{ck}.parquet')))

pd.concat(all_pred, ignore_index=True).to_parquet(os.path.join(OUTPUT_DIR, 'predictions_all_claims.parquet'), index=False)
pd.concat(all_val, ignore_index=True).to_parquet(os.path.join(OUTPUT_DIR, 'validation_all_claims.parquet'), index=False)
pd.concat(all_gs, ignore_index=True).to_parquet(os.path.join(OUTPUT_DIR, 'shap_global_all_claims.parquet'), index=False)
pd.concat(all_fi, ignore_index=True).to_parquet(os.path.join(OUTPUT_DIR, 'feature_importance_all_claims.parquet'), index=False)

summary_df = pd.DataFrame(all_model_summaries)
summary_df.to_parquet(os.path.join(OUTPUT_DIR, 'model_summary.parquet'), index=False)
print("Combined files saved.")

## 5. Model Summary

In [ ]:
summary_df = pd.DataFrame(all_model_summaries)
display(summary_df[['claim_type', 'prediction_month', 'train_rows', 'val_rows', 'predict_rows',
                     'train_positive', 'val_positive', 'cv_auc_mean', 'val_auc', 'val_f1']])

## 6. Cross-Claim SHAP Comparison

In [ ]:
combined_shap = pd.read_parquet(os.path.join(OUTPUT_DIR, 'shap_global_all_claims.parquet'))
for claim in CLAIM_TYPES:
    subset = combined_shap[combined_shap['claim_type']==claim].nlargest(10, 'mean_norm_shap_pct')
    print(f"\n{claim.upper()} - Top 10 by normalized SHAP %:")
    for _, row in subset.iterrows():
        bar = '█' * int(row['mean_norm_shap_pct'])
        print(f"  {row['mean_norm_shap_pct']:6.2f}%  {bar}  {row['feature']}")

## 7. Inspect High-Risk Patient (Falls)

In [ ]:
pred = pd.read_parquet(os.path.join(OUTPUT_DIR, 'predictions_fall.parquet'))
shap_n = pd.read_parquet(os.path.join(OUTPUT_DIR, 'shap_normalized_fall.parquet'))

top_idx = pred['y_proba'].idxmax()
top = pred.loc[top_idx]
print(f"Highest risk patient for FALL (2025-02):")
print(f"  Resident: {top['resident_id']}")
print(f"  Probability: {top['y_proba']:.4f}")
print(f"  Actual: {'CLAIM' if top['y_true']==1 else 'No claim'}")

patient_shap = shap_n.loc[top_idx].drop(['resident_id','facility_id','year_month','claim_type'])
top_feats = patient_shap.nlargest(15)
print(f"\nTop 15 contributing features:")
for feat, val in top_feats.items():
    bar = '█' * int(val)
    print(f"  {val:6.2f}%  {bar}  {feat}")